# 🧠 Training the Smart Compose Brain

**Chapter 02 — Gmail Smart Compose**

> **Goal:** Understand how we take a blank Transformer and turn it into a Smart Compose model — covering pretraining, finetuning, next-token prediction, decoding strategies, evaluation metrics, and the full production system.

> **Vibe:** You built the brain in notebook 01. Now let's TRAIN it, TEST it, and SHIP it. 🚀

## 📚 Two-Stage Training: Pretrain Then Finetune

### The Analogy That Makes It Click

Imagine hiring a new employee for your email support team:

**Stage 1 — Pretraining (Learning English)**
> You don't start by teaching them email templates. You hire someone who already speaks fluent English — they've read millions of books, news articles, Wikipedia pages. They understand grammar, sentence structure, context, and vocabulary. This took years and is incredibly expensive.

**Stage 2 — Finetuning (Learning Email Writing)**
> Now you take your fluent English speaker and give them 3 months of on-the-job training specifically on professional emails. They learn the specific style, common phrases, how to respond to different email types. This is much cheaper — they already know English!

```
PRETRAINING                          FINETUNING
─────────────────                    ─────────────────────────
Data: Trillions of tokens            Data: Billions of email tokens
      (books, web, Wikipedia)              (anonymized Gmail data)

Cost: 💰💰💰💰💰 Very expensive       Cost: 💰 Much cheaper

Time: Weeks/months on 1000s GPUs     Time: Days on 100s GPUs

Goal: Learn GENERAL language         Goal: Learn EMAIL-SPECIFIC patterns
      patterns and world knowledge         "Dear ...", "Thanks for..."
                                           "Best regards, ..."

Task: Next-token prediction          Task: Same! Next-token prediction
      on general text                      but on email text
```

### Why Does This Work?

Pretraining gives the model a "feature extractor" for language that transfers to email writing. The Transformer layers learn UNIVERSAL representations of meaning — they just need to be nudged toward email-specific patterns via finetuning.

This is **transfer learning** at massive scale, and it's why training language models isn't as expensive as it looks — most of the cost happens once during pretraining, then every product (Smart Compose, Google Translate, etc.) just needs a relatively cheap finetune.

> 🎯 **Interview gold:** "We use a two-stage training approach: general pretraining on large text corpora (next-token prediction), followed by domain-specific finetuning on anonymized email data. This is more efficient than training from scratch for emails, and pretraining gives the model world knowledge that helps with context (recognizing names, dates, topics) even during email completion."

## 🎯 The Training Objective: Next-Token Prediction

### What the Model Actually Learns

Both pretraining and finetuning use the SAME objective: given a sequence of tokens, **predict the next token**. It's shockingly simple — and shockingly powerful.

```
Training example: "Thanks for the report"

INPUT (what the model sees):    OUTPUT (what we want predicted):
─────────────────────────────   ────────────────────────────────
[BOS] Thanks for the            → report
[BOS] Thanks for                → the
[BOS] Thanks                    → for
[BOS]                           → Thanks

One sentence generates 4 training examples! 🎉
This is called TEACHER FORCING — feed the true previous tokens,
predict the next one.
```

### Why Is This Objective So Powerful?

To predict the next word well, the model MUST learn:
- **Grammar**: "The cat ___ on the mat" → needs a verb
- **Facts**: "The capital of France is ___" → Paris
- **Style**: "Dear Dr. Smith, I am writing to ___" → email style
- **Context**: "I'll review it and get back to you by ___" → a day of the week

The model is FORCED to understand language deeply just to do next-token prediction well. That's the magic of self-supervised learning — the labels come from the data itself, so we can use unlimited text!

```
Why it scales:
  Text corpus:  The entire internet
  Labels:       The next word (free! already in the data!)
  
  → Zero annotation cost, unlimited training data 🎉
```

In [ ]:
"""
🎯 Next-token prediction: train a tiny Transformer on a few email sentences

We'll see exactly how the training data is constructed and how
the model learns to predict the next token.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# ============================================================
# Build a tiny vocabulary and dataset
# ============================================================
sentences = [
    "thanks for the report",
    "thanks for getting back to me",
    "please find the attached document",
    "looking forward to our meeting",
    "let me know if you have questions",
    "i will review the report",
    "please review the attached document",
    "thanks for your help",
]

# Build vocabulary
all_words = []
for sent in sentences:
    all_words.extend(sent.split())

word_counts = Counter(all_words)
vocab = ['[PAD]', '[BOS]', '[EOS]'] + sorted(word_counts.keys())
token_to_idx = {w: i for i, w in enumerate(vocab)}
idx_to_token = {i: w for w, i in token_to_idx.items()}

VOCAB_SIZE = len(vocab)
PAD_IDX = token_to_idx['[PAD]']
BOS_IDX = token_to_idx['[BOS]']
EOS_IDX = token_to_idx['[EOS]']

print("=" * 60)
print("📚 TINY EMAIL VOCABULARY")
print("=" * 60)
print(f"Vocab size: {VOCAB_SIZE}")
print(f"Vocab: {vocab}")

# ============================================================
# Tokenize and create training data
# ============================================================
def tokenize(sentence):
    tokens = [BOS_IDX] + [token_to_idx[w] for w in sentence.split()] + [EOS_IDX]
    return torch.tensor(tokens)

tokenized = [tokenize(s) for s in sentences]

# Pad to same length for batching
max_len = max(len(t) for t in tokenized)
padded = torch.stack([
    F.pad(t, (0, max_len - len(t)), value=PAD_IDX)
    for t in tokenized
])

print(f"\n{'='*60}")
print(f"NEXT-TOKEN PREDICTION TRAINING DATA")
print(f"{'='*60}")
print(f"Sentence: '{sentences[0]}'")
print(f"Tokenized: {tokenized[0].tolist()}")
print(f"Tokens:    {[idx_to_token[i] for i in tokenized[0].tolist()]}")
print(f"\nTraining pairs from this sentence:")
tokens = tokenized[0].tolist()
for i in range(len(tokens)-1):
    context = [idx_to_token[t] for t in tokens[:i+1]]
    target = idx_to_token[tokens[i+1]]
    print(f"  Input: {context} → Target: '{target}'")

# ============================================================
# Define tiny Transformer (reusing components from notebook 01)
# ============================================================
def positional_encoding(max_seq_len, d_model):
    PE = torch.zeros(max_seq_len, d_model)
    positions = torch.arange(max_seq_len).unsqueeze(1).float()
    div_term = torch.pow(10000.0, torch.arange(0, d_model, 2).float() / d_model)
    PE[:, 0::2] = torch.sin(positions / div_term)
    PE[:, 1::2] = torch.cos(positions / div_term)
    return PE

class TinyTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=32, num_heads=2, d_ff=64,
                 num_layers=1, max_len=32):
        super().__init__()
        self.d_model = d_model
        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.register_buffer('PE', positional_encoding(max_len, d_model))
        encoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=num_heads, dim_feedforward=d_ff,
            batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerDecoder(encoder_layer, num_layers=num_layers)
        self.lm_head = nn.Linear(d_model, vocab_size)
        self.lm_head.weight = self.embed.weight  # weight tying
    
    def forward(self, x):
        seq_len = x.size(1)
        emb = self.embed(x) + self.PE[:seq_len].unsqueeze(0)
        # Causal mask
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(seq_len, device=x.device)
        # For decoder-only: use same tensor as memory (self-attention trick)
        out = self.transformer(emb, emb, tgt_mask=tgt_mask,
                               memory_mask=tgt_mask,
                               tgt_is_causal=True)
        return self.lm_head(out)

# ============================================================
# Train the tiny model
# ============================================================
torch.manual_seed(42)
model = TinyTransformer(VOCAB_SIZE)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)

# Training: input = tokens[:-1], target = tokens[1:]
inputs = padded[:, :-1]   # (batch, seq_len-1)
targets = padded[:, 1:]   # (batch, seq_len-1)

print(f"\n{'='*60}")
print(f"TRAINING TINY TRANSFORMER")
print(f"{'='*60}")

losses = []
NUM_EPOCHS = 200

for epoch in range(NUM_EPOCHS):
    model.train()
    optimizer.zero_grad()
    
    logits = model(inputs)  # (batch, seq_len-1, vocab_size)
    
    # Flatten for cross-entropy: ignore PAD tokens
    logits_flat = logits.reshape(-1, VOCAB_SIZE)
    targets_flat = targets.reshape(-1)
    
    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=PAD_IDX)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS}  |  Loss: {loss.item():.4f}")

# Plot training curve
plt.figure(figsize=(10, 4))
plt.plot(losses, color='#2196F3', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Cross-Entropy Loss', fontsize=12)
plt.title('📉 Training Loss Curve — Tiny Smart Compose', fontsize=14)
plt.grid(True, alpha=0.3)
plt.annotate(f'Final loss: {losses[-1]:.3f}', xy=(len(losses)-1, losses[-1]),
            xytext=(len(losses)*0.7, losses[0]*0.6),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=11, color='red')
plt.tight_layout()
plt.show()

print(f"\n✅ Training complete! Loss: {losses[0]:.3f} → {losses[-1]:.3f}")
print(f"   Lower loss = model is LESS surprised by the correct next token")

In [ ]:
"""
📉 Cross-Entropy Loss Explained: The Score Card for Language Models

Cross-entropy measures how much the model's predicted probability distribution
matches the true distribution (which is always 1 for the correct token, 0 for rest).
"""
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# ============================================================
# Visual explanation of cross-entropy
# ============================================================
print("=" * 60)
print("📉 CROSS-ENTROPY LOSS EXPLAINED")
print("=" * 60)

print("""
Setup: The correct next word is 'report' (index 5).
Three models make different predictions:
""")

# Example: 8-word vocabulary, correct answer is index 5 ('report')
vocab_example = ['[PAD]', '[BOS]', 'thanks', 'for', 'the', 'report', 'meeting', 'email']
correct_idx = 5  # 'report'

# Three model predictions (logits → probabilities)
scenarios = {
    '🟢 Confident & Correct': torch.tensor([0.1, 0.1, 0.1, 0.1, 0.1, 5.0, 0.1, 0.1]),
    '🟡 Uncertain (spread)':  torch.tensor([0.5, 0.5, 0.5, 0.5, 0.5, 0.6, 0.5, 0.5]),
    '🔴 Confident & Wrong':   torch.tensor([0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 5.0, 0.1]),
}

print(f"{'Scenario':<35} {'P(correct)':>12} {'Cross-Entropy Loss':>20} {'Interpretation'}")
print("-" * 95)

for name, logits in scenarios.items():
    probs = F.softmax(logits, dim=-1)
    p_correct = probs[correct_idx].item()
    
    # Cross-entropy = -log(P(correct))
    ce_loss = F.cross_entropy(logits.unsqueeze(0), torch.tensor([correct_idx])).item()
    
    if ce_loss < 0.5:
        interp = "✅ Great prediction!"
    elif ce_loss < 2.0:
        interp = "⚠️  Decent but unsure"
    else:
        interp = "❌ Very wrong!"
    
    print(f"{name:<35} {p_correct:>12.4f} {ce_loss:>20.4f} {interp}")

print(f"""
Formula: Cross-Entropy = -log( P(correct token) )

  P(correct) = 0.99  →  Loss = -log(0.99) ≈ 0.01  (nearly perfect! 🎉)
  P(correct) = 0.50  →  Loss = -log(0.50) ≈ 0.69  (random guess)
  P(correct) = 0.01  →  Loss = -log(0.01) ≈ 4.61  (terrible! 😱)

The loss becomes VERY high when the model is CONFIDENT but WRONG.
This teaches the model: 'Be careful about certainty!'
""")

# ============================================================
# Visualize how cross-entropy behaves as a function of P(correct)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: CE loss vs P(correct)
p_values = np.linspace(0.001, 0.999, 1000)
ce_values = -np.log(p_values)

axes[0].plot(p_values, ce_values, color='#2196F3', linewidth=2.5)
axes[0].fill_between(p_values, ce_values, alpha=0.1, color='#2196F3')
axes[0].axvline(x=0.5, color='gray', linestyle='--', alpha=0.5, label='Random guess (P=0.5)')
axes[0].scatter([0.99, 0.5, 0.01], [-np.log(0.99), -np.log(0.5), -np.log(0.01)],
               c=['green', 'orange', 'red'], s=100, zorder=5)
axes[0].annotate('P=0.99 → Loss≈0.01\n(confident & correct)', xy=(0.99, -np.log(0.99)),
               xytext=(0.7, 1), arrowprops=dict(arrowstyle='->', color='green'), fontsize=9)
axes[0].annotate('P=0.01 → Loss≈4.6\n(confident & wrong!)', xy=(0.01, -np.log(0.01)),
               xytext=(0.2, 3.5), arrowprops=dict(arrowstyle='->', color='red'), fontsize=9)
axes[0].set_xlabel('P(correct token)', fontsize=12)
axes[0].set_ylabel('Cross-Entropy Loss', fontsize=12)
axes[0].set_title('📉 Cross-Entropy vs Probability\nof the Correct Token', fontsize=12)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 6)
axes[0].legend()

# Plot 2: Bar chart of probability distributions for our 3 scenarios
bar_width = 0.25
x_pos = np.arange(len(vocab_example))
colors_bar = ['#4CAF50', '#FF9800', '#F44336']

for i, (name, logits) in enumerate(scenarios.items()):
    probs = F.softmax(logits, dim=-1).numpy()
    bars = axes[1].bar(x_pos + i*bar_width, probs, bar_width,
                       label=name[:15], alpha=0.8, color=colors_bar[i])

axes[1].axvline(x=correct_idx + bar_width, color='black', linewidth=2,
               linestyle='--', label=f"Correct: '{vocab_example[correct_idx]}'")
axes[1].set_xticks(x_pos + bar_width)
axes[1].set_xticklabels(vocab_example, rotation=45, ha='right', fontsize=9)
axes[1].set_ylabel('Probability', fontsize=12)
axes[1].set_title('📊 Predicted Probability Distributions\nfor 3 Model Scenarios', fontsize=12)
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('Cross-Entropy Loss: The Report Card for Language Models 📝',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("🎯 KEY INSIGHT: Cross-entropy loss = -log(probability of the right answer)")
print("   Training pushes the model to assign HIGH probability to the correct token!")

## 🔍 Decoding Strategy 1: Greedy Search

Once the model is trained, how do we actually generate a suggestion?

The simplest approach: at each step, just pick the **most likely next token**. Then use that as input and pick the next most likely, and so on.

```
GREEDY SEARCH for "Thanks for the ___"

Step 1: P(next | "Thanks for the")
   report  → 0.42  ← pick this! ✅
   email   → 0.18
   meeting → 0.11
   ...

Step 2: P(next | "Thanks for the report")
   .       → 0.38  ← pick this! ✅ (end of phrase)
   ,       → 0.25
   and     → 0.15

Result: "Thanks for the report."
```

### Greedy Pros and Cons

| ✅ Pros | ❌ Cons |
|---|---|
| Super fast: O(1) per step | Suboptimal: locally greedy ≠ globally best |
| Deterministic | Can get stuck in repetition loops |
| Easy to implement | Misses better completions down other paths |

**The core problem:** Choosing the best next token at each step doesn't guarantee the best overall sequence.

```
Example where greedy fails:

  "I would like to ___"

  Greedy picks: "schedule" (p=0.35)
  → "I would like to schedule the"
  → "I would like to schedule the best" ← awkward!

  But 'meet' (p=0.28) leads to:
  → "I would like to meet"
  → "I would like to meet with you" ← much better!

  Greedy got trapped by its first choice!
```

In [ ]:
"""
🔍 Greedy search decoding — complete implementation
"""
import torch
import torch.nn.functional as F

def greedy_decode(model, prompt_tokens, token_to_idx, idx_to_token,
                  max_new_tokens=5, eos_idx=None):
    """
    Greedy decoding: at each step, pick the highest-probability next token.
    
    Args:
        model: trained language model
        prompt_tokens: tensor of input token indices
        max_new_tokens: max tokens to generate
        eos_idx: stop if we generate this token
    Returns:
        generated_tokens: list of generated token indices
        all_steps: list of (token_str, prob) for each step (for display)
    """
    model.eval()
    generated = []
    all_steps = []
    
    current_input = prompt_tokens.unsqueeze(0)  # (1, seq_len)
    
    with torch.no_grad():
        for step in range(max_new_tokens):
            # Forward pass
            logits = model(current_input)  # (1, seq_len, vocab_size)
            
            # Look at the LAST token's prediction (predict what comes next)
            next_token_logits = logits[0, -1, :]  # (vocab_size,)
            next_token_probs = F.softmax(next_token_logits, dim=-1)
            
            # GREEDY: pick the token with highest probability
            next_token = next_token_probs.argmax().item()
            next_token_prob = next_token_probs[next_token].item()
            
            token_str = idx_to_token.get(next_token, '[UNK]')
            all_steps.append((token_str, next_token_prob, next_token_probs))
            generated.append(next_token)
            
            # Stop if we hit EOS
            if eos_idx and next_token == eos_idx:
                break
            
            # Append the new token and continue
            next_token_tensor = torch.tensor([[next_token]])
            current_input = torch.cat([current_input, next_token_tensor], dim=1)
    
    return generated, all_steps


# ============================================================
# Demo greedy decoding with our trained model
# ============================================================
print("=" * 60)
print("🔍 GREEDY SEARCH DECODING")
print("=" * 60)

prompts = [
    "thanks for",
    "please find",
    "looking forward",
]

for prompt in prompts:
    # Tokenize prompt
    tokens = [BOS_IDX] + [
        token_to_idx.get(w, token_to_idx.get('[PAD]', 0))
        for w in prompt.split()
    ]
    prompt_tensor = torch.tensor(tokens)
    
    generated, steps = greedy_decode(
        model, prompt_tensor, token_to_idx, idx_to_token,
        max_new_tokens=4, eos_idx=EOS_IDX
    )
    
    generated_str = ' '.join(
        t for t, _, _ in steps
        if t not in ['[EOS]', '[PAD]', '[BOS]']
    )
    
    print(f"\nPrompt:  '{prompt}'")
    print(f"Output:  '{prompt} {generated_str}'")
    print(f"Steps:")
    for step_num, (tok, prob, all_probs) in enumerate(steps):
        top3_vals, top3_ids = torch.topk(all_probs, 3)
        top3 = [(idx_to_token.get(i.item(), '?'), p.item())
                for i, p in zip(top3_ids, top3_vals)]
        print(f"  Step {step_num+1}: selected '{tok}' (p={prob:.3f})  "
              f"| top-3: {[f'{t}:{p:.2f}' for t, p in top3]}")

print("\n" + "=" * 60)
print("⚠️  GREEDY FAILURE CASE (contrived example):")
print("=" * 60)
print("""
Imagine: P("I would like to schedule" | context) seems good step-by-step,
but leads to an awkward completion. Meanwhile:

  "meet" has lower probability at step 1...
  BUT P("meet with you" | "I would like to meet") is very natural!

Greedy never explores the 'meet' path — it committed too early.
This is exactly the problem Beam Search solves! 👇
""")

## 🔦 Decoding Strategy 2: Beam Search

Instead of committing to ONE path (greedy), beam search keeps the **top-K most promising partial sequences** at each step, where K is the **beam width**.

```
BEAM SEARCH (beam_width = 3)
Prompt: "Thanks for the"

Step 1 — Expand from start:
  Keep top-3 candidates:
    ① "Thanks for the report"   score = log(0.42) = -0.87
    ② "Thanks for the email"    score = log(0.18) = -1.71
    ③ "Thanks for the meeting"  score = log(0.11) = -2.21

Step 2 — Expand each candidate:
  From ①: "report" can be followed by:
    → ① + "."   score = -0.87 + log(0.38) = -0.87 + (-0.97) = -1.84
    → ① + ","   score = -0.87 + log(0.25) = -0.87 + (-1.39) = -2.26
    
  From ②: "email" can be followed by:
    → ② + "."   score = -1.71 + log(0.55) = -1.71 + (-0.60) = -2.31
    ...
    
  From ③: "meeting" can be followed by:
    → ③ + " on" score = -2.21 + log(0.60) = -2.21 + (-0.51) = -2.72
    ...

  Keep top-3 overall:
    ① "Thanks for the report."  score = -1.84  ← BEST!
    ② "Thanks for the report,"  score = -2.26
    ③ "Thanks for the email."   score = -2.31

Final output: ① "Thanks for the report." (highest log-probability!)
```

### Why Beam Search for Smart Compose?

- **Deterministic**: same input → same output (important for professional emails!)
- **Better than greedy**: explores multiple paths, avoids early commitment mistakes
- **Controllable**: beam_width is a quality-vs-compute tradeoff knob
- **No surprises**: unlike random sampling, it won't generate weird/creative outputs

> 🎯 **Interview gold:** "For Smart Compose we use beam search (not temperature sampling) because we want the **most likely** completion, not a **diverse/creative** one. A chatbot benefits from diversity — it shouldn't always say the same thing. But email completion should be professional and predictable. Beam search with width 4-8 gives high-quality deterministic suggestions."

| When to use | Strategy |
|---|---|
| Professional email completion | Beam search |
| Creative writing assistant | Temperature sampling / top-p |
| Machine translation | Beam search |
| Chatbot responses | Top-k or nucleus (top-p) sampling |
| Code completion | Beam search or greedy |

In [ ]:
"""
🔦 Beam Search from scratch
"""
import torch
import torch.nn.functional as F
import math

def beam_search_decode(model, prompt_tokens, idx_to_token,
                       beam_width=3, max_new_tokens=5, eos_idx=None,
                       length_penalty=0.7):
    """
    Beam search decoding.
    
    At each step, maintain 'beam_width' candidate sequences.
    Score each sequence by the SUM of log-probabilities.
    Apply length penalty to avoid preferring short sequences.
    
    Args:
        model: trained language model
        prompt_tokens: tensor of input token indices  (seq_len,)
        beam_width: how many candidates to keep (K)
        max_new_tokens: max tokens to generate
        eos_idx: stop when this token is generated
        length_penalty: penalize shorter sequences (0 = no penalty, 1 = strong)
    Returns:
        best_sequence: list of generated token indices
        all_beams: all candidate beams with scores (for display)
    """
    model.eval()
    
    # Each beam: (log_score, token_ids_list, is_complete)
    beams = [(0.0, prompt_tokens.tolist(), False)]
    completed_beams = []
    
    with torch.no_grad():
        for step in range(max_new_tokens):
            if not beams:
                break
            
            all_candidates = []
            
            for log_score, token_ids, is_complete in beams:
                if is_complete:
                    completed_beams.append((log_score, token_ids, True))
                    continue
                
                # Forward pass
                input_tensor = torch.tensor(token_ids).unsqueeze(0)
                logits = model(input_tensor)  # (1, seq_len, vocab_size)
                
                # Next-token probabilities
                next_log_probs = F.log_softmax(logits[0, -1, :], dim=-1)
                
                # Expand: try all tokens, keep top beam_width
                top_log_probs, top_token_ids = torch.topk(next_log_probs, beam_width)
                
                for new_log_prob, new_token in zip(top_log_probs, top_token_ids):
                    new_token = new_token.item()
                    new_log_score = log_score + new_log_prob.item()
                    new_sequence = token_ids + [new_token]
                    is_done = (eos_idx is not None and new_token == eos_idx)
                    all_candidates.append((new_log_score, new_sequence, is_done))
            
            # Sort all candidates by score (accounting for length penalty)
            def score_with_length_penalty(candidate):
                log_score, tokens, _ = candidate
                # Length penalty: longer sequences get a small bonus
                gen_len = len(tokens) - len(prompt_tokens)
                penalty = ((5 + max(gen_len, 1)) / 6) ** length_penalty
                return log_score / penalty
            
            all_candidates.sort(key=score_with_length_penalty, reverse=True)
            
            # Keep top beam_width candidates
            beams = all_candidates[:beam_width]
    
    # Add remaining beams to completed
    completed_beams.extend(beams)
    
    # Return the best completed beam
    completed_beams.sort(key=lambda x: x[0] / max(len(x[1]) - len(prompt_tokens), 1),
                        reverse=True)
    
    best_score, best_tokens, _ = completed_beams[0]
    # Return only the generated tokens (not the prompt)
    generated = best_tokens[len(prompt_tokens.tolist()):]
    
    return generated, completed_beams


# ============================================================
# Compare greedy vs beam search
# ============================================================
print("=" * 60)
print("🔦 BEAM SEARCH DECODING (vs Greedy)")
print("=" * 60)

for prompt in ["thanks for", "please find"]:
    prompt_words = prompt.split()
    tokens = [BOS_IDX] + [
        token_to_idx.get(w, token_to_idx.get('[PAD]', 0))
        for w in prompt_words
    ]
    prompt_tensor = torch.tensor(tokens)
    
    # Greedy
    gen_greedy, greedy_steps = greedy_decode(
        model, prompt_tensor, token_to_idx, idx_to_token,
        max_new_tokens=4, eos_idx=EOS_IDX
    )
    greedy_str = ' '.join(
        t for t, _, _ in greedy_steps if t not in ['[EOS]', '[PAD]', '[BOS]']
    )
    
    # Beam search with different beam widths
    print(f"\n{'─'*55}")
    print(f"Prompt: '{prompt}'")
    print(f"{'─'*55}")
    print(f"  Greedy (beam=1):  '{prompt} {greedy_str}'")
    
    for bw in [2, 3, 4]:
        gen_beam, all_beams = beam_search_decode(
            model, prompt_tensor, idx_to_token,
            beam_width=bw, max_new_tokens=4, eos_idx=EOS_IDX
        )
        beam_str = ' '.join(
            idx_to_token.get(t, '[?]') for t in gen_beam
            if idx_to_token.get(t, '') not in ['[EOS]', '[PAD]', '[BOS]']
        )
        print(f"  Beam (width={bw}):   '{prompt} {beam_str}'")
    
    # Show all beam candidates
    print(f"\n  All beam candidates (width=3), sorted by score:")
    for rank, (score, tokens_list, _) in enumerate(all_beams[:4]):
        gen_tokens = tokens_list[len(prompt_tensor):]
        gen_words = ' '.join(
            idx_to_token.get(t, '?') for t in gen_tokens
            if idx_to_token.get(t, '') not in ['[EOS]', '[PAD]', '[BOS]']
        )
        marker = " ← chosen" if rank == 0 else ""
        print(f"    #{rank+1} score={score:.3f}: '{prompt} {gen_words}'{marker}")

print("\n" + "=" * 60)
print("🎯 KEY INSIGHT: Beam search explores multiple paths!")
print("   In Smart Compose: beam_width=4 gives a good quality/speed tradeoff")
print("   Larger beam = better quality but slower (quadratic search space)")

## 📊 Evaluation: How Do We Know If Smart Compose Is Good?

Two types of metrics: **offline** (can evaluate before shipping) and **online** (requires real user data).

### Offline Metric 1: Perplexity 🤔

Perplexity measures how **"surprised"** the model is by real text. Lower perplexity = better model.

```
INTUITION:

  A perfect model that always assigns P=1 to the correct token:
    → Perplexity = 1  (never surprised)

  A random model over 30,000 tokens:
    → Perplexity = 30,000  (maximally surprised)

  Smart Compose target:
    → Perplexity < 20  (usually knows what's coming!)

FORMULA:

  Perplexity = exp( - (1/N) × Σ log P(token_i | context_i) )

  = exp( average cross-entropy loss over the test set )

  If average cross-entropy = 2.0 nats:
    Perplexity = e^2.0 ≈ 7.4  (pretty good!)

  If average cross-entropy = 5.0 nats:
    Perplexity = e^5.0 ≈ 148.4  (not great)
```

### Offline Metric 2: ExactMatch@N 🎯

"Does the model's top-N suggestions contain exactly what the user actually typed?"

```
ExactMatch@3 example:

  User typed: "Thanks for the report. I'll review"
  User's actual next phrase: "it tomorrow"
  
  Model's top-3 suggestions:
    #1: "it and respond"       ← doesn't match
    #2: "it tomorrow"          ← MATCH! ✅ (rank 2)
    #3: "it by Friday"         ← doesn't match
  
  This counts as ExactMatch@3 = 1 (a match was in top 3)
  ExactMatch@1 = 0 (match was not #1)
```

**Target for Smart Compose:** ExactMatch@3 > 30%

In [ ]:
"""
📊 Calculate perplexity on a small test example
"""
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

def calculate_perplexity(model, token_ids, pad_idx=0):
    """
    Calculate perplexity of a model on a given token sequence.
    
    Perplexity = exp(average cross-entropy loss)
    
    Args:
        model: trained language model
        token_ids: (seq_len,) tensor of token indices
        pad_idx: padding token index (excluded from calculation)
    Returns:
        perplexity: float
        per_token_surprisal: list of -log P(token_i | context) for each token
    """
    model.eval()
    
    with torch.no_grad():
        input_ids = token_ids[:-1].unsqueeze(0)  # (1, seq_len-1)
        target_ids = token_ids[1:]               # (seq_len-1,)
        
        logits = model(input_ids)  # (1, seq_len-1, vocab_size)
        log_probs = F.log_softmax(logits[0], dim=-1)  # (seq_len-1, vocab_size)
        
        # Per-token surprisal = -log P(correct_token | context)
        per_token_surprisal = []
        total_log_prob = 0.0
        count = 0
        
        for i, target in enumerate(target_ids):
            if target.item() == pad_idx:
                continue
            surprisal = -log_probs[i, target].item()
            per_token_surprisal.append(surprisal)
            total_log_prob += -surprisal  # log prob is negative surprisal
            count += 1
        
        avg_nll = -total_log_prob / count if count > 0 else float('inf')
        perplexity = math.exp(avg_nll)
    
    return perplexity, per_token_surprisal


# ============================================================
# Calculate perplexity on test sentences
# ============================================================
# Test on both seen (training) and novel sentences
test_cases = [
    ("thanks for the report", "seen during training"),
    ("please find the attached document", "seen during training"),
    ("thanks for the meeting notes", "mostly novel — 'notes' not in training"),
    ("please review the attached report", "combination of training words"),
]

print("=" * 65)
print("📊 PERPLEXITY EVALUATION")
print("=" * 65)
print(f"{'Sentence':<45} {'PPL':>7} {'Type'}")
print("-" * 80)

all_ppls = []
all_surprisals = []
all_labels = []

for sentence, label in test_cases:
    # Tokenize (handle unknowns)
    tokens = [BOS_IDX] + [
        token_to_idx.get(w, token_to_idx.get('[PAD]', 0))
        for w in sentence.split()
    ] + [EOS_IDX]
    token_tensor = torch.tensor(tokens)
    
    ppl, surprisals = calculate_perplexity(model, token_tensor, pad_idx=PAD_IDX)
    all_ppls.append(ppl)
    all_surprisals.append(surprisals)
    all_labels.append(sentence)
    
    flag = "✅" if ppl < 10 else ("⚠️" if ppl < 50 else "❌")
    print(f"{flag} {sentence:<43} {ppl:>7.1f}  {label}")

print(f"\nFormula: Perplexity = exp(average_negative_log_prob)")
print(f"         Low PPL → model is confident and correct")
print(f"         High PPL → model is surprised by the text")

# ============================================================
# Visualize per-token surprisal for one sentence
# ============================================================
sentence_to_viz = "thanks for the report"
idx_to_viz = 0
tokens_for_viz = sentence_to_viz.split()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Per-token surprisal
surprisals_viz = all_surprisals[idx_to_viz]
tokens_viz_display = ['[BOS]'] + tokens_for_viz  # what each surprisal corresponds to

colors = ['#4CAF50' if s < 1.5 else ('#FF9800' if s < 3.0 else '#F44336')
          for s in surprisals_viz]

bars = axes[0].bar(range(len(surprisals_viz)), surprisals_viz, color=colors, alpha=0.8)
axes[0].set_xticks(range(len(surprisals_viz)))
axes[0].set_xticklabels(tokens_viz_display[:len(surprisals_viz)],
                         rotation=45, ha='right', fontsize=11)
axes[0].set_ylabel('-log P(token | context)', fontsize=12)
axes[0].set_title(f'Per-Token Surprisal\n"{sentence_to_viz}"', fontsize=12)
axes[0].axhline(y=np.mean(surprisals_viz), color='black', linestyle='--',
               label=f'Mean = {np.mean(surprisals_viz):.2f}', linewidth=2)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, surprisals_viz):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                f'{val:.2f}', ha='center', va='bottom', fontsize=9)

# Plot 2: PPL comparison across test sentences
short_labels = [s[:20] + '...' if len(s) > 20 else s for s in all_labels]
bar_colors = ['#4CAF50' if p < 10 else ('#FF9800' if p < 50 else '#F44336')
              for p in all_ppls]
axes[1].barh(short_labels, all_ppls, color=bar_colors, alpha=0.8)
axes[1].axvline(x=20, color='red', linestyle='--', linewidth=2,
               label='Target: PPL < 20')
axes[1].set_xlabel('Perplexity', fontsize=12)
axes[1].set_title('Perplexity by Sentence\n(lower = better)', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='x')
for i, ppl in enumerate(all_ppls):
    axes[1].text(ppl + 0.3, i, f'{ppl:.1f}', va='center', fontsize=10)

plt.suptitle('📊 Perplexity: How Surprised Is the Model? 🤔',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n🎯 KEY: Low perplexity on training sentences ✅")
print(f"        Higher perplexity on novel phrases — expected!")

In [ ]:
"""
🎯 ExactMatch@N — Does the suggestion match what was actually typed?
"""
import torch
import torch.nn.functional as F

def generate_top_n_suggestions(model, prompt_tokens, idx_to_token,
                                n=3, max_new_tokens=4, eos_idx=None):
    """
    Generate top-N suggestion phrases using beam search.
    Returns the N candidates as strings.
    """
    model.eval()
    _, all_beams = beam_search_decode(
        model, prompt_tokens, idx_to_token,
        beam_width=n, max_new_tokens=max_new_tokens,
        eos_idx=eos_idx
    )
    
    # Get the top-N unique suggestions
    suggestions = []
    seen = set()
    
    for score, tokens_list, _ in all_beams:
        gen_tokens = tokens_list[len(prompt_tokens):]
        words = [idx_to_token.get(t, '[?]') for t in gen_tokens
                 if idx_to_token.get(t, '') not in ['[EOS]', '[PAD]', '[BOS]']]
        suggestion = ' '.join(words)
        if suggestion and suggestion not in seen:
            suggestions.append((suggestion, score))
            seen.add(suggestion)
        if len(suggestions) >= n:
            break
    
    return suggestions


def exact_match_at_n(model, test_examples, token_to_idx, idx_to_token,
                     n=3, max_new_tokens=4):
    """
    Calculate ExactMatch@N on a test set.
    
    test_examples: list of (prompt, ground_truth_continuation) tuples
    """
    hits = 0
    results = []
    
    for prompt, ground_truth in test_examples:
        # Tokenize prompt
        tokens = [BOS_IDX] + [
            token_to_idx.get(w, token_to_idx.get('[PAD]', 0))
            for w in prompt.split()
        ]
        prompt_tensor = torch.tensor(tokens)
        
        # Get top-N suggestions
        suggestions = generate_top_n_suggestions(
            model, prompt_tensor, idx_to_token,
            n=n, max_new_tokens=max_new_tokens, eos_idx=EOS_IDX
        )
        
        # Check if ground truth appears in top-N
        match = any(
            ground_truth.lower().strip() == sug.lower().strip()
            for sug, _ in suggestions
        )
        if match:
            hits += 1
        
        results.append({
            'prompt': prompt,
            'ground_truth': ground_truth,
            'suggestions': [s for s, _ in suggestions],
            'match': match
        })
    
    exact_match = hits / len(test_examples) if test_examples else 0
    return exact_match, results


# ============================================================
# Evaluate ExactMatch@N
# ============================================================

# Test set: (prompt, what_user_actually_typed)
test_examples = [
    ("thanks for", "the report"),
    ("thanks for", "getting back to me"),
    ("please find", "the attached document"),
    ("looking forward", "to our meeting"),
    ("let me know", "if you have questions"),
    ("i will", "review the report"),
    ("thanks for your", "help"),
    ("please", "review the attached document"),
]

print("=" * 65)
print("🎯 EXACTMATCH@N EVALUATION")
print("=" * 65)

for n in [1, 2, 3]:
    em_score, results = exact_match_at_n(
        model, test_examples, token_to_idx, idx_to_token, n=n
    )
    print(f"ExactMatch@{n}: {em_score:.1%}  ({int(em_score*len(test_examples))}/{len(test_examples)} matches)")

# Detailed results for ExactMatch@3
_, detailed_results = exact_match_at_n(
    model, test_examples, token_to_idx, idx_to_token, n=3
)

print(f"\n{'─'*65}")
print(f"DETAILED RESULTS (ExactMatch@3):")
print(f"{'─'*65}")
for r in detailed_results:
    status = "✅ HIT" if r['match'] else "❌ MISS"
    print(f"\n{status}")
    print(f"  Prompt:       '{r['prompt']}'")
    print(f"  Ground truth: '{r['ground_truth']}'")
    print(f"  Suggestions:  {r['suggestions']}")

print(f"\n{'─'*65}")
print("💡 In production Smart Compose:")
print("  ExactMatch@3 > 30%  →  the suggestion appears within top-3 often enough")
print("  Acceptance rate > 10% →  users actually TAB to accept suggestions")
print("  EM measures quality; acceptance rate measures UX relevance")

## 🏗️ Full Smart Compose System Design

The model is just one piece of the puzzle. Here's how the complete production system works:

```
FULL SMART COMPOSE PRODUCTION SYSTEM
═══════════════════════════════════════════════════════════════

USER ACTION: Types "Thanks for the" in Gmail
                   │
                   ▼
         ┌─────────────────┐
         │  CLIENT (Gmail) │  Sends request on keystroke (debounced ~150ms)
         │  context:       │  Includes:
         │  - email body   │    • Typed text so far
         │  - subject line │    • Subject line
         │  - recipient    │    • Recipient (for personalization)
         │  - timestamp    │    • Time of day
         └────────┬────────┘
                  │  API request (<50ms budget to backend)
                  ▼
         ┌─────────────────────────────────────────────────┐
         │              SMART COMPOSE BACKEND              │
         │                                                 │
         │  ┌──────────────────────────────────────────┐  │
         │  │  1. TRIGGERING SERVICE  🚦               │  │
         │  │                                          │  │
         │  │  Lightweight classifier (fast!):         │  │
         │  │  • Is cursor at end of a word/phrase?    │  │
         │  │  • Is typing speed slow (pausing)?       │  │
         │  │  • Is the partial sentence grammatical?  │  │
         │  │  • Did user recently reject a suggestion?│  │
         │  │                                          │  │
         │  │  IF confidence < threshold → STOP here   │  │
         │  │  (Don't waste GPU compute on bad moments)│  │
         │  └──────────────────┬───────────────────────┘  │
         │     confidence OK   │                           │
         │                     ▼                           │
         │  ┌──────────────────────────────────────────┐  │
         │  │  2. PHRASE GENERATOR  🧠                 │  │
         │  │                                          │  │
         │  │  Inputs: tokenized context              │  │
         │  │  Model: decoder-only Transformer        │  │
         │  │         ~20M parameters, 6 layers       │  │
         │  │                                          │  │
         │  │  Decoding: beam search (width=4)        │  │
         │  │  Stop: EOS token OR max_len (8 tokens)  │  │
         │  │                                          │  │
         │  │  KV Cache: reuse computed keys/values   │  │
         │  │  across keystrokes (HUGE speedup!)      │  │
         │  └──────────────────┬───────────────────────┘  │
         │                     │ candidate phrase          │
         │                     ▼                           │
         │  ┌──────────────────────────────────────────┐  │
         │  │  3. POST-PROCESSING SERVICE  🧹          │  │
         │  │                                          │  │
         │  │  Safety filters:                        │  │
         │  │  • Toxicity classifier (remove hate)    │  │
         │  │  • PII detector (no SSNs, credit cards) │  │
         │  │  • Off-topic filter (is it about email?)│  │
         │  │                                          │  │
         │  │  Quality filters:                       │  │
         │  │  • Grammar check                        │  │
         │  │  • Minimum confidence threshold         │  │
         │  │  • Length: 3-8 words (not too short/long│  │
         │  └──────────────────┬───────────────────────┘  │
         └─────────────────────┼───────────────────────────┘
                               │  suggestion (or empty)
                               ▼
         ┌─────────────────────────────────┐
         │  CLIENT (Gmail)                 │
         │                                 │
         │  Show gray suggestion text:     │
         │  "Thanks for the report. I'll" │
         │                                 │
         │  Press Tab → ACCEPT (log event) │
         │  Keep typing → REJECT           │
         └─────────────────────────────────┘

KEY OPTIMIZATIONS:
  • Quantization: FP32 → INT8 (4x smaller model, ~3x faster inference)
  • KV Caching: reuse attention key-value pairs between keystrokes
  • Request batching: process multiple users' requests together on GPU
  • Edge deployment option: run model on-device (no network latency!)
  • Model distillation: train small model to mimic large teacher model

LATENCY BUDGET: Total < 100ms
  Network round-trip:  ~20ms
  Triggering service:  ~5ms
  Transformer inference: ~60ms (with INT8 + KV cache)
  Post-processing:     ~10ms
  Safety filter:       ~5ms
═══════════════════════════════════════════════════════════════
```

## 🎤 Interview Walkthrough: Talking Points Cheat Sheet

### The 30-Second Pitch

> "Smart Compose is a real-time text completion system. We frame it as **autoregressive next-token prediction** using a **decoder-only Transformer** (~20M params for speed). Training is two-stage: pretrain on large text, finetune on email data. At inference, we use **beam search** for deterministic, high-quality suggestions. The system has three main components: a lightweight **triggering service** that decides when to show suggestions, the **phrase generator** (the Transformer with beam search), and a **post-processing filter** for safety and quality. The hard constraint is **<100ms latency**, which drives every architectural decision: small model, quantization, KV caching."

---

### Cheat Sheet: Answering Common Follow-Ups

| Question | Key Points to Hit |
|---|---|
| **Why Transformer, not RNN?** | (1) Self-attention = O(1) path between any positions, not O(n) like RNN. (2) Parallel computation = faster training. (3) No vanishing gradient on long context. |
| **Why decoder-only, not encoder-decoder?** | We're doing language modeling (left-to-right text generation), not seq2seq. Decoder-only is simpler, faster to serve, and naturally suited for next-token prediction. GPT architecture! |
| **Why BPE tokenization?** | Sweet spot: handles OOV via subword decomposition, vocab ~30K (manageable), shorter sequences than char-level. Word-level can't handle new words at all. |
| **Why beam search, not sampling?** | Email completion needs to be professional and deterministic. Beam search gives the most likely completion. Random sampling is better for chatbots (diversity) but bad for autocomplete. |
| **How to handle <100ms latency?** | (1) Small model ~20M params. (2) INT8 quantization. (3) KV cache across keystrokes. (4) Request batching. (5) Edge/on-device inference option. (6) Triggering service to avoid wasting compute. |
| **How to evaluate offline?** | Perplexity (how surprised by test text), ExactMatch@N (does top-N match what user typed). Then online: acceptance rate (Tab presses), keystroke savings. |
| **How to personalize?** | Condition on recipient, subject, time-of-day. Federated learning for on-device personalization without sending emails to servers. Recent email context as few-shot examples. |
| **What about privacy?** | Federated learning: model trains on-device, only gradient updates sent to server (not raw emails). Differential privacy on gradients. No storing email content centrally. |
| **How to deploy/monitor?** | A/B test new model versions. Monitor: acceptance rate, latency p50/p99, toxicity rate. Automatic rollback if acceptance rate drops or safety violations spike. |
| **How to handle cold start (new user)?** | Use global model as default. As user accepts/rejects suggestions, fine-tune the on-device model (federated). Personalization improves over time. |

---

### Interview Anti-Patterns to Avoid 🚫

1. **Jumping to architecture before requirements** — Always clarify latency, scale, personalization first
2. **Proposing GPT-4 scale models** — You'll be shot down immediately. Justify the ~20M param choice!
3. **Forgetting the triggering service** — A common miss. The model shouldn't fire on every keystroke!
4. **Saying "just use beam search" without explaining tradeoffs** — Why not greedy? Why not sampling? Show you understand the tradeoffs.
5. **Ignoring the latency constraint** — It drives EVERYTHING: model size, quantization, caching, edge deployment
6. **Only mentioning offline metrics** — Always also mention online metrics and the gap between them

---

### The Five Numbers to Memorize 🔢

```
  < 100ms   total end-to-end latency requirement
  ~20M      model parameter count (tiny vs GPT-3's 175B)
  ~30K      BPE vocabulary size
  beam=4    typical beam search width
  >30%      ExactMatch@3 target for good quality
```

## 🧪 Quiz Time! Test Your Training & Sampling Knowledge

---

### Question 1: Training Objective

Why is next-token prediction considered "self-supervised" rather than "supervised" learning?

- A) Because it doesn't use labels at all
- B) Because the labels (next tokens) come from the data itself — no human annotation needed
- C) Because it uses reinforcement learning rewards
- D) Because the model supervises its own training without a loss function

<details>
<summary>🔓 Click to reveal answer</summary>

**B)** Self-supervised means the labels are derived from the data itself. For next-token prediction, given a text corpus, the input is all tokens up to position t, and the label is token at position t+1 — no human needed to annotate anything. This is the key insight that enabled scaling to the entire internet: unlimited training data with zero annotation cost. Compare to supervised learning where a human must label each example (e.g., "this email is formal" vs "this email is casual").

</details>

---

### Question 2: Cross-Entropy Loss

A language model assigns probability P=0.001 to the correct next token (it's very wrong!). What is approximately the cross-entropy loss for this prediction?

- A) 0.001
- B) 0.999
- C) 6.9 (since -log(0.001) ≈ 6.9)
- D) 1000

<details>
<summary>🔓 Click to reveal answer</summary>

**C)** Cross-entropy = -log(P(correct)) = -log(0.001) = -(-6.908) ≈ 6.9. The negative log function makes the loss VERY high when the model is confident and wrong (P→0 means loss→∞). This is intentional: the model is punished heavily for being confidently wrong, which teaches calibration. For reference: perfect model (P=1.0) → loss = 0; random model over 30K vocab (P=1/30000 ≈ 0.000033) → loss ≈ 10.4.

</details>

---

### Question 3: Beam Search

You're building Smart Compose and considering increasing the beam width from 4 to 16. What's the primary tradeoff?

- A) Higher beam width improves quality but increases latency (more candidates to evaluate)
- B) Higher beam width reduces latency because the model can stop earlier
- C) Higher beam width reduces quality because it considers too many options
- D) Beam width has no effect on quality, only on diversity

<details>
<summary>🔓 Click to reveal answer</summary>

**A)** Quality vs latency tradeoff! Wider beam = more candidate paths explored = better chance of finding the globally optimal sequence. But at each step, you must run the model on beam_width sequences instead of 1, multiplying compute roughly by beam_width. For Smart Compose with a strict 100ms budget, beam_width=4 is typically chosen as a sweet spot. If you have more compute budget (e.g., offline suggestion generation), you could use beam_width=8 or 16 for better quality.

</details>

---

### Question 4: Perplexity vs ExactMatch

Your team trains two models. Model A has perplexity 15. Model B has perplexity 25. But in A/B testing, users accept Model B's suggestions more often. How is this possible?

- A) This is impossible — lower perplexity always means better suggestions
- B) Perplexity measures average prediction quality, but ExactMatch/acceptance is about specific high-confidence suggestions. Model B might be less certain overall but more accurate at high-confidence moments
- C) Model A is overfitting
- D) The A/B test is measuring the wrong thing

<details>
<summary>🔓 Click to reveal answer</summary>

**B)** This is the offline-online metric gap! Perplexity averages over ALL tokens, including hard-to-predict ones (names, specific dates, technical jargon). A model can have great average perplexity but poor performance at the specific moments users need suggestions. Model B might have learned to be more confident and accurate at "complete a sentence" moments — common email patterns — even if it's worse on rare vocabulary. This is why ExactMatch and especially online acceptance rate are the north star metrics, with perplexity as a health check rather than the ultimate goal.

</details>

---

### Question 5: System Design

The Smart Compose team notices that showing suggestions too aggressively (on every keystroke) reduces the acceptance rate. What's the right architectural solution?

- A) Use a larger Transformer model so suggestions are always correct
- B) Add a lightweight triggering service that uses a classifier to decide WHEN to show suggestions
- C) Increase beam search width to generate better suggestions
- D) Switch from beam search to random sampling for more diverse suggestions

<details>
<summary>🔓 Click to reveal answer</summary>

**B)** The triggering service! Model quality is orthogonal to WHEN to show suggestions. Even a perfect model will annoy users if it fires mid-word or when users are typing quickly and don't want interruptions. The triggering classifier is typically a lightweight model (logistic regression or tiny MLP) that looks at: cursor position, typing speed, grammaticality of the partial sentence, user's recent accept/reject behavior. It gates the expensive Transformer model — if triggering says "not now," we never even call the Transformer. This saves compute AND improves user experience. This is often the most overlooked component in system design interviews!

</details>